# Extended Wait Prediction

This notebook starts machine-learning work for the ED patient flow project. It uses the cleaned CSV for the first modelling pass.

The model predicts extended ED waiting time using only arrival-time or near-arrival features. Post-arrival outcomes are excluded to avoid leakage.

## Scope Boundaries

- Primary target: `extended_wait_2hr_flag`
- Secondary comparison target: `long_wait_4hr_flag`
- Allowed feature theme: information available at or near arrival
- Excluded leakage fields: wait time, visit length, admission, observation, departure outcomes, and ED death outcome
- SQL integration point: replace the CSV source with `vw_ml_wait_features` after that view is available

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/processed/nhamcs_2022_visits_clean.csv")
MODEL_OUTPUT_DIR = Path("../outputs/models")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, na_values=["NULL", "", " "])
df.shape

In [ ]:
validation_summary = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_visit_ids": int(df["visit_id"].duplicated().sum()),
    "valid_wait_times": int(df["wait_time_minutes"].notna().sum()),
    "extended_wait_2hr": int(df["extended_wait_2hr_flag"].sum()),
    "long_wait_4hr": int(df["long_wait_4hr_flag"].sum()),
}
validation_summary

## Target Choice

The 2-hour target is the first modelling target because it has more positive cases than the 4-hour target. The 4-hour flag remains useful as a stricter operational KPI, but it is much more imbalanced.

In [ ]:
target_balance = (
    df[["extended_wait_2hr_flag", "long_wait_4hr_flag"]]
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "positive_cases", "mean": "positive_rate"})
)
target_balance["positive_rate"] = target_balance["positive_rate"] * 100
target_balance

## Feature Set

These features are limited to information known at arrival or very close to arrival. The model intentionally avoids outcomes that are only known after the ED visit has progressed.

In [ ]:
target = "extended_wait_2hr_flag"

candidate_features = [
    "visit_month",
    "visit_day",
    "arrival_hour",
    "age_years",
    "sex",
    "residence_type",
    "arrival_by_ambulance",
    "ambulance_transfer",
    "triage_level",
    "pain_scale",
    "pulse_bpm",
    "respiratory_rate",
    "systolic_bp",
    "diastolic_bp",
    "oxygen_saturation",
    "temperature_f",
    "chronic_condition_count",
    "region",
    "metropolitan_status",
]

leakage_fields = [
    "wait_time_minutes",
    "visit_length_minutes",
    "left_without_being_seen",
    "left_before_treatment_complete",
    "left_against_medical_advice",
    "died_in_ed",
    "admitted_to_hospital",
    "observation_then_hospitalized",
    "observation_then_discharged",
    "admission_destination",
]

assert not set(candidate_features).intersection(leakage_fields)

model_df = df[candidate_features + [target]].dropna(subset=[target]).copy()
model_df[target] = model_df[target].astype(int)
model_df.shape

In [ ]:
numeric_features = model_df[candidate_features].select_dtypes(include="number").columns.tolist()
categorical_features = [col for col in candidate_features if col not in numeric_features]

numeric_features, categorical_features

## Baseline and First Model

The baseline predicts the majority class for every visit. The final comparison includes balanced logistic regression and balanced Random Forest models.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = model_df[candidate_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

baseline_model = DummyClassifier(strategy="most_frequent")
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=250,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
    }

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_test)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_test, probabilities)
    else:
        metrics["roc_auc"] = None

    return metrics

results = [
    evaluate_model("majority_class_baseline", baseline_model, X_train, X_test, y_train, y_test),
    evaluate_model("logistic_regression_balanced", logistic_model, X_train, X_test, y_train, y_test),
    evaluate_model("random_forest_balanced", random_forest_model, X_train, X_test, y_train, y_test),
]

metrics_df = pd.DataFrame(results)
metrics_df

In [ ]:
try:
    import joblib

    metrics_df.to_csv(MODEL_OUTPUT_DIR / "wait_prediction_model_metrics.csv", index=False)
    best_model_name = metrics_df.sort_values(["f1", "roc_auc"], ascending=False).iloc[0]["model"]
    best_model = {
        "logistic_regression_balanced": logistic_model,
        "random_forest_balanced": random_forest_model,
    }.get(best_model_name, logistic_model)
    joblib.dump(best_model, MODEL_OUTPUT_DIR / "wait_prediction_model.joblib")
except ImportError:
    print("joblib is not installed; metrics were saved but the fitted model was not exported.")

MODEL_OUTPUT_DIR